In [1]:
import os
from dotenv import load_dotenv, find_dotenv, dotenv_values
from agents import Agent, Runner, Tool, WebSearchTool, trace, function_tool
from agents.mcp import MCPServerStdio
import instructions


In [2]:
# Locate .env in this directory or any parent directory
dotenv_path = find_dotenv()
if not dotenv_path:
    raise FileNotFoundError('.env not found in repository or parent directories')

# Load into os.environ (preserves existing variables unless overridden by .env)
load_dotenv(dotenv_path, override=False)
# Also read raw values as a dict (useful to expose into notebook globals)
env = {k: v for k, v in dotenv_values(dotenv_path).items() if v is not None}

# Export into notebook globals for easy access by name
globals().update(env)

print('Loaded .env from', dotenv_path)
print('Loaded keys:', list(env.keys()))

Loaded .env from /media/nathan/linux_ssd/github/agentic_ai_trip_planner/.env
Loaded keys: ['OPENAI_API_KEY', 'GROQ_API_KEY', 'PUSHOVER_USER', 'PUSHOVER_TOKEN', 'SENDGRID_API_KEY', 'GOOGLE_API_KEY', 'SERPER_API_KEY', 'LANGSMITH_TRACING', 'LANGSMITH_ENDPOINT', 'LANGSMITH_API_KEY', 'LANGSMITH_PROJECT', 'POLYGON_API_KEY', 'POLYGON_PLAN', 'BRAVE_API_KEY']


In [ ]:
# A simple agent to plan a basic trip...

duration = "5 days"
destination = "Paris"
activities = ["sightseeing", "dining", "arcades", "museums"]

web_search_tool = WebSearchTool(search_context_size="low") #This tool can add costs to the agent

trip_planner_agent = Agent(
    name="Trip Planner Agent",
    instructions="An agent that helps users plan trips by searching for destinations, accommodations, and activities.",
    #instructions=f"plan a {duration} trip to {destination} including {', '.join(activities)}. Use the web search tool to find relevant information.",
    tools=[
        web_search_tool
    ]
)

directions = f"plan a {duration} trip to {destination} including {', '.join(activities)}. Use the web search tool to find relevant information."

with trace("Trip Planner Agent"):
    result = await Runner.run(trip_planner_agent, directions)
    print(result.final_output)




Here’s a 5‑day Paris trip itinerary combining sightseeing, museums, dining, and arcades. All details below are based on up‑to‑date sources — enjoy planning!

---

**Day 1: Historic Landmarks & Elegant Dining**  
• Begin at the Louvre — home of the Mona Lisa, Venus de Milo, and more. Be sure to book tickets in advance.([ouistars.com](https://www.ouistars.com/blog/paris-museums-art-guide-2025-best-exhibitions-galleries-cultural-experiences?utm_source=openai))  
• Walk through the Tuileries Gardens to Palais Royal and stroll the exquisite covered passages with boutique cafés and glass‑roof arcades.([dualconvert.com](https://dualconvert.com/discover-paris-france-5-day-paris-itinerary/?utm_source=openai))  
• Lunch with views at Café Marly by the Louvre Pyramid, or choose a cozy brasserie in the 1st arrondissement.([dualconvert.com](https://dualconvert.com/discover-paris-france-5-day-paris-itinerary/?utm_source=openai))  
• Evening: Seine river cruise or sunset from Montmartre (Sacré‑Cœur).

[non-fatal] Tracing: server error 503, retrying.


## Now go and look at the trace

https://platform.openai.com/traces

In [ ]:
# Add in a tool to save the output to a .md file


duration = "5 days"
destination = "Paris"
activities = ["sightseeing", "dining", "arcades", "museums"]

@function_tool
def save_to_md(content: str, filename: str):
    """Save content to a markdown file in the output directory."""
    os.makedirs("output", exist_ok=True)
    with open(f"output/{filename}.md", "w") as f:
        f.write(content)
    return f"Saved trip plan to output/{filename}.md"

#save_tool = Tool(save_to_md)

web_search_tool = WebSearchTool(search_context_size="low") #This tool can add costs to the agent

trip_planner_agent = Agent(
    name="Trip Planner Agent",
    instructions="An agent that helps users plan trips by searching for destinations, accommodations, and activities. Use the save tool to save the final plan in markdown format.",
    #instructions=f"plan a {duration} trip to {destination} including {', '.join(activities)}. Use the web search tool to find relevant information.",
    tools=[
        web_search_tool,
        save_to_md
    ]
)

directions = f"plan a {duration} trip to {destination} including {', '.join(activities)}. Use the web search tool to find relevant information. Then save the complete trip plan to a markdown file named 'trip_plan'."

with trace("Trip Planner Agent"):
    result = await Runner.run(trip_planner_agent, directions)
    print(result.final_output)

Your 5-day Paris itinerary—featuring sightseeing, dining, arcades, and top museums—has been researched using the latest info and saved in a markdown file named trip_plan.md. This plan covers current closures, special exhibitions, and must-see locations for an enriched Paris experience!

If you’d like, I can provide the plan as text here, or guide you on downloading and using your markdown file for your journey.


In [ ]:
#now, let's add a filesystem tool to read and write files in a sandboxed environment

sandbox_path = os.path.abspath(os.path.join(os.getcwd(), "output"))
print(f"Using sandbox path: {sandbox_path}")
files_params = {"command": "npx", "args": ["-y", "@modelcontextprotocol/server-filesystem", sandbox_path]}

async with MCPServerStdio(params=files_params,client_session_timeout_seconds=60) as server:
    file_tools = await server.list_tools()

print(file_tools)


web_search_tool = WebSearchTool(search_context_size="low") #This tool can add costs to the agent
print(web_search_tool)


directions = f"plan a {duration} trip to {destination} including {', '.join(activities)}. Use the web search tool to find relevant information. Then save the complete trip plan to a markdown file named 'trip_plan'."

async with MCPServerStdio(params=files_params, client_session_timeout_seconds=60) as mcp_server_files:
    trip_planner_agent = Agent(
        name="Trip Planner Agent",
        instructions="An agent that helps users plan trips by searching for destinations, accommodations, and activities. Use the write_file tool to save the final plan in markdown format.",
        tools=[web_search_tool],
        mcp_servers=[mcp_server_files]
    )
    with trace("Trip Planner Agent"):
        result = await Runner.run(trip_planner_agent, directions)
        print(result.final_output)



Using sandbox path: /media/nathan/linux_ssd/github/agentic_ai_trip_planner/openai_agents_sdk/output
[Tool(name='read_file', title='Read File (Deprecated)', description='Read the complete contents of a file as text. DEPRECATED: Use read_text_file instead.', inputSchema={'$schema': 'http://json-schema.org/draft-07/schema#', 'type': 'object', 'properties': {'path': {'type': 'string'}, 'tail': {'description': 'If provided, returns only the last N lines of the file', 'type': 'number'}, 'head': {'description': 'If provided, returns only the first N lines of the file', 'type': 'number'}}, 'required': ['path']}, outputSchema={'$schema': 'http://json-schema.org/draft-07/schema#', 'type': 'object', 'properties': {'content': {'type': 'string'}}, 'required': ['content'], 'additionalProperties': False}, icons=None, annotations=ToolAnnotations(title=None, readOnlyHint=True, destructiveHint=None, idempotentHint=None, openWorldHint=None), meta=None, execution=ToolExecution(taskSupport='forbidden')), T

In [ ]:
# Add in a tool to save the output to a .md file

duration = "5 days"
destination = "Paris"
activities = ["sightseeing", "dining", "museums"]

@function_tool
def save_to_disk(content: str, filename: str):
    """Save content to a markdown file in the output directory."""
    os.makedirs("output", exist_ok=True)
    with open(f"output/{filename}.md", "w") as f:
        f.write(content)
    return f"Saved trip plan to output/{filename}.md"

@function_tool
def read_file_from_disk(filename: str):
    """Read content from a markdown file in the output directory."""
    with open(f"output/{filename}.md", "r") as f:
        content = f.read()
    return content

web_search_tool = WebSearchTool(search_context_size="low") #This tool can add costs to the agent

trip_planner_agent = Agent(
    name="Trip Planner Agent",
    instructions="An agent that helps users plan trips by searching for destinations, accommodations, and activities.",
    tools=[
        web_search_tool,
        save_to_disk,
        read_file_from_disk
    ]
)

print(instructions.trip_planner_instructions)

with trace("Trip Planner Agent"):
    result = await Runner.run(trip_planner_agent, instructions.trip_planner_instructions)
    print(result.final_output)

You are a methodical and detail-oriented trip planning assistant. Your task is to create a COMPLETE, timeline-based trip itinerary with specific departure/arrival times and durations for every activity.

CRITICAL: You must complete the ENTIRE itinerary before finishing. Do not stop at research phase. Do not ask for permission to continue. Work through all steps until you have a fully detailed day-by-day schedule.

The customer has provided the following details for their trip:
- Home Location: Columbus, OH
- Departure Date: 2026/02/25
- Return Date: 2026/03/07
- Destination: Tokyo, Japan
- Must-Do Activities: Visit the Tokyo Tower, Explore Akihabara, Experience a traditional tea ceremony, Visit the Tsukiji Fish Market, Take a day trip to Mount Fuji   
- Number of Travelers: 3
- Ages of Travelers: 51, 50, 17
- Other Considerations: 
  - I will be running in the Tokyo marathon on Sunday March 1, so I only need a relaxing place to eat on that day.
  - Starting on March 3, throughout the r